In [35]:
from hmmlearn import hmm
import numpy as np

In [36]:
# Sirayla:
# States
# Baslangıc olasiliklari
# Gecisl olasiliklari
# Emisyon olasiliklari

states = ["e", "v"] # State

ilk_prob = {"e": 1.0, "v": .0} # ilk hep e olcak diyoruz

gecis_prob = {
    "e": {"e": .6, "v": .4},
    "v": {"e": .2, "v": .8}
} # P(yeni_hidden|eski_hidden) olasiliklari bunlar hidden arasi

emisyon_prob = {
    "e": {"High": 0.7, "Low": 0.3},
    "v": {"High": 0.1, "Low": 0.9}
} # Hiddendan gozleme gecis olasiliklari birnevi transition matrixim

In [37]:
# Ornekler
ornek_data = ["High", "Low"]

v = [{}]  # Olasılık matrisi
path = {} # En olası yollar burda sadece 2 hidden var 2 de path uzunlugu 2^2 bakilcak

In [38]:
# P(harf|s0)
for s in states:
    v[0][s] = ilk_prob[s] * emisyon_prob[s][ornek_data[0]]
    path[s] = [s]

print(v)

[{'e': 0.7, 'v': 0.0}]


In [39]:
# Path uzunlugunda don
for t in range(1, len(ornek_data)):
    v.append({})
    newpath = {}

    for s in states:
        # (Onceki durumun ihtimali * Oncekinden buraya gecme ihtimali * Burdan gozleme gecme ihtimali)-> her ihtimal icin hesaplar maksimumu alir
        # Burasi biraz ters oldu aslinda prev_s disa da olur
        (prob, state) = max((v[t - 1][prev_s] * gecis_prob[prev_s][s] * emisyon_prob[s][ornek_data[t]], prev_s) for prev_s in states)
        v[t][s] = prob
        newpath[s] = path[state] + [s] # state icinde en yuksek ihtimalli onceki state var s de gecisini kontrol ettigim state

    path = newpath

n = len(ornek_data) - 1
(max_prob, final_state) = max((v[n][s], s) for s in states)

print("En olasi dizisi: ", path[final_state])
print("Olasilik: ", max_prob)

En olasi dizisi:  ['e', 'v']
Olasilik:  0.252


In [40]:
path

{'e': ['e', 'e'], 'v': ['e', 'v']}

In [41]:
veri = {"High": 1, "Low": 0}

# Rastgele veri
ev_sequences = [
    np.array([[1], [0], [1]]),
    np.array([[1], [1], [0]]),
]
okul_sequences = [
    np.array([[0], [1], [0], [0]]),
    np.array([[0], [0], [1], [0]]),
]

# Model olusturma
def hmm_(veri, n_states):
    lengths = [len(seq) for seq in veri]
    X = np.concatenate(veri)
    model = hmm.MultinomialHMM(n_components=n_states, n_iter=100, random_state=42)
    model.fit(X, lengths)
    return model

# HMM
ev_model = hmm_(ev_sequences, n_states=2)
okul_model = hmm_(okul_sequences, n_states=3)

MultinomialHMM has undergone major changes. The previous version was implementing a CategoricalHMM (a special case of MultinomialHMM). This new implementation follows the standard definition for a Multinomial distribution (e.g. as in https://en.wikipedia.org/wiki/Multinomial_distribution). See these issues for details:
https://github.com/hmmlearn/hmmlearn/issues/335
https://github.com/hmmlearn/hmmlearn/issues/340
MultinomialHMM has undergone major changes. The previous version was implementing a CategoricalHMM (a special case of MultinomialHMM). This new implementation follows the standard definition for a Multinomial distribution (e.g. as in https://en.wikipedia.org/wiki/Multinomial_distribution). See these issues for details:
https://github.com/hmmlearn/hmmlearn/issues/335
https://github.com/hmmlearn/hmmlearn/issues/340


In [42]:
# Test data
test_observation = np.array([[1], [0], [1]])

# Skor hesaplama
ev_score = ev_model.score(test_observation)
okul_score = okul_model.score(test_observation)

print("EV model skor: ", ev_score)
print("OKUL model skor: ", okul_score)

if ev_score > okul_score:
    print("EV")
else:
    print("OKUL")

EV model skor:  -2.7755575615628914e-17
OKUL model skor:  -1.6653345369377348e-16
EV
